# Marshall digital: corpo de prova virtual

Método de Awuah, Garcia-Hernandez & Valentin (2024), *Construction and Building Materials* 416, 135281.

**Como usar:** edite a célula **Mistura** e depois use *Ambiente de execução → Executar tudo*.
O teste de 250 g leva ~5 min; o corpo de prova Marshall completo, ~20–30 min.
No fim, o navegador baixa o visualizador 3D (HTML) e os STL.


In [ ]:
!pip -q install scikit-image pillow


In [ ]:
%%writefile dem.c
// DEM de aglomerados rígidos de esferas (multi-sphere) num molde Marshall cilíndrico.
// Unidades: mm, kg, s  (força em mN).  Compilar: gcc -O3 -march=native -shared -fPIC dem.c -o libdem.so -lm
#include <math.h>
#include <stdlib.h>
#include <string.h>
#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif
#ifdef _WIN32
#define EXPORT __declspec(dllexport)
#else
#define EXPORT
#endif

typedef struct {
    int np;              // partículas
    int *sp_start, *sp_n;   // índice das esferas de cada partícula
    double *sp_loc;      // esferas em coordenadas do corpo (x,y,z,r) já escaladas
    double *pos, *q, *vel, *w;   // estado (np*3, np*4, np*3, np*3) — w no referencial global
    double *mass, *Ib;   // massa e inércia principal (np*3)
    double *brad;        // raio envolvente
    double R;            // raio do molde
    double zp;           // posição do pistão
} Sys;

static double g_coat = 0.0;
EXPORT void set_coat(double c){ g_coat = c; }
static double *spw = NULL; static int nsp_tot = 0;
static int *pairs = NULL; static int npairs = 0, cap_pairs = 0;
static double *pos_at_build = NULL;

static void rotate(const double *q, const double *v, double *o) {
    // o = q v q*
    double w = q[0], x = q[1], y = q[2], z = q[3];
    double tx = 2*(y*v[2]-z*v[1]), ty = 2*(z*v[0]-x*v[2]), tz = 2*(x*v[1]-y*v[0]);
    o[0] = v[0] + w*tx + (y*tz - z*ty);
    o[1] = v[1] + w*ty + (z*tx - x*tz);
    o[2] = v[2] + w*tz + (x*ty - y*tx);
}
static void rotate_inv(const double *q, const double *v, double *o) {
    double qc[4] = {q[0], -q[1], -q[2], -q[3]}; rotate(qc, v, o);
}

static void build_pairs(Sys *S, double skin) {
    int n = S->np; double bmax = 0;
    for (int i = 0; i < n; i++) if (S->brad[i] > bmax) bmax = S->brad[i];
    double cs = 2*bmax + skin;
    double zmin = 1e9, zmax = -1e9;
    for (int i = 0; i < n; i++) { double z = S->pos[3*i+2]; if (z<zmin) zmin=z; if (z>zmax) zmax=z; }
    double x0 = -S->R - cs, y0 = -S->R - cs, z0 = zmin - cs;
    int nx = (int)((2*S->R + 2*cs)/cs) + 1, ny = nx, nz = (int)((zmax - zmin + 2*cs)/cs) + 1;
    int ncell = nx*ny*nz;
    int *head = malloc(sizeof(int)*ncell), *next = malloc(sizeof(int)*n), *cell = malloc(sizeof(int)*n);
    for (int c = 0; c < ncell; c++) head[c] = -1;
    for (int i = 0; i < n; i++) {
        int cx = (int)((S->pos[3*i]-x0)/cs), cy = (int)((S->pos[3*i+1]-y0)/cs), cz = (int)((S->pos[3*i+2]-z0)/cs);
        if (cx<0) cx=0; if (cx>=nx) cx=nx-1; if (cy<0) cy=0; if (cy>=ny) cy=ny-1; if (cz<0) cz=0; if (cz>=nz) cz=nz-1;
        int c = (cz*ny + cy)*nx + cx; cell[i] = c; next[i] = head[c]; head[c] = i;
    }
    npairs = 0;
    for (int i = 0; i < n; i++) {
        int c = cell[i], cx = c % nx, cy = (c/nx) % ny, cz = c/(nx*ny);
        for (int dz=-1; dz<=1; dz++) for (int dy=-1; dy<=1; dy++) for (int dx=-1; dx<=1; dx++) {
            int X=cx+dx, Y=cy+dy, Z=cz+dz;
            if (X<0||Y<0||Z<0||X>=nx||Y>=ny||Z>=nz) continue;
            for (int j = head[(Z*ny+Y)*nx+X]; j >= 0; j = next[j]) {
                if (j <= i) continue;
                double ddx = S->pos[3*i]-S->pos[3*j], ddy = S->pos[3*i+1]-S->pos[3*j+1], ddz = S->pos[3*i+2]-S->pos[3*j+2];
                double rc = S->brad[i] + S->brad[j] + 2*g_coat + skin;
                if (ddx*ddx+ddy*ddy+ddz*ddz < rc*rc) {
                    if (npairs >= cap_pairs) { cap_pairs = cap_pairs ? 2*cap_pairs : 1<<16; pairs = realloc(pairs, sizeof(int)*2*cap_pairs); }
                    pairs[2*npairs] = i; pairs[2*npairs+1] = j; npairs++;
                }
            }
        }
    }
    free(head); free(next); free(cell);
    if (!pos_at_build) pos_at_build = malloc(sizeof(double)*3*n);
    memcpy(pos_at_build, S->pos, sizeof(double)*3*n);
}

static void contact(double *F, double *T, int a, int b, const double *ca, double ra, const double *cb, double rb,
                    Sys *S, double kn, double zeta, double mu, double *maxov) {
    // b = -1 → parede com centro virtual cb e massa infinita
    double d[3] = {ca[0]-cb[0], ca[1]-cb[1], ca[2]-cb[2]};
    double dist = sqrt(d[0]*d[0]+d[1]*d[1]+d[2]*d[2]);
    double ov = ra + rb - dist; if (ov <= 0 || dist < 1e-12) return;
    if (ov > *maxov) *maxov = ov;
    double n[3] = {d[0]/dist, d[1]/dist, d[2]/dist};
    double xc[3] = {ca[0]-n[0]*(ra-ov/2), ca[1]-n[1]*(ra-ov/2), ca[2]-n[2]*(ra-ov/2)};
    double *pa = &S->pos[3*a], *va = &S->vel[3*a], *wa = &S->w[3*a];
    double raV[3] = {xc[0]-pa[0], xc[1]-pa[1], xc[2]-pa[2]};
    double v[3] = {va[0] + wa[1]*raV[2]-wa[2]*raV[1], va[1] + wa[2]*raV[0]-wa[0]*raV[2], va[2] + wa[0]*raV[1]-wa[1]*raV[0]};
    double meff = S->mass[a];
    double rbV[3] = {0,0,0};
    if (b >= 0) {
        double *pb = &S->pos[3*b], *vb = &S->vel[3*b], *wb = &S->w[3*b];
        rbV[0]=xc[0]-pb[0]; rbV[1]=xc[1]-pb[1]; rbV[2]=xc[2]-pb[2];
        v[0] -= vb[0] + wb[1]*rbV[2]-wb[2]*rbV[1];
        v[1] -= vb[1] + wb[2]*rbV[0]-wb[0]*rbV[2];
        v[2] -= vb[2] + wb[0]*rbV[1]-wb[1]*rbV[0];
        meff = S->mass[a]*S->mass[b]/(S->mass[a]+S->mass[b]);
    }
    double vn = v[0]*n[0]+v[1]*n[1]+v[2]*n[2];
    double cn = 2*zeta*sqrt(meff*kn);
    double fn = kn*ov - cn*vn; if (fn < 0) fn = 0;
    double vt[3] = {v[0]-vn*n[0], v[1]-vn*n[1], v[2]-vn*n[2]};
    double vtm = sqrt(vt[0]*vt[0]+vt[1]*vt[1]+vt[2]*vt[2]);
    double ft = 0;
    if (vtm > 1e-12) { ft = cn*vtm; if (ft > mu*fn) ft = mu*fn; ft /= vtm; }
    double f[3] = {fn*n[0]-ft*vt[0], fn*n[1]-ft*vt[1], fn*n[2]-ft*vt[2]};
    F[3*a]+=f[0]; F[3*a+1]+=f[1]; F[3*a+2]+=f[2];
    T[3*a]+=raV[1]*f[2]-raV[2]*f[1]; T[3*a+1]+=raV[2]*f[0]-raV[0]*f[2]; T[3*a+2]+=raV[0]*f[1]-raV[1]*f[0];
    if (b >= 0) {
        F[3*b]-=f[0]; F[3*b+1]-=f[1]; F[3*b+2]-=f[2];
        T[3*b]-=rbV[1]*f[2]-rbV[2]*f[1]; T[3*b+1]-=rbV[2]*f[0]-rbV[0]*f[2]; T[3*b+2]-=rbV[0]*f[1]-rbV[1]*f[0];
    }
}

// Executa nsteps. Pistão: se piston_mode==1 desce com velocidade vp até zp_target.
// Retorna em out[]: [0]=máx sobreposição, [1]=força no pistão (mN), [2]=energia cinética, [3]=z topo, [4]=zp
EXPORT int run(int np, int *sp_start, int *sp_n, double *sp_loc, double *pos, double *q, double *vel, double *w,
        double *mass, double *Ib, double *brad, double R, double *zp_io, double zp_target, double vp,
        int nsteps, double dt, double g, double kn, double zeta, double mu, double gdamp, double shake_amp,
        double shake_freq, double t0, double *out) {
    Sys S = {np, sp_start, sp_n, sp_loc, pos, q, vel, w, mass, Ib, brad, R, *zp_io};
    int ns = sp_start[np-1] + sp_n[np-1];
    if (ns != nsp_tot) { spw = realloc(spw, sizeof(double)*4*ns); nsp_tot = ns; }
    double *F = malloc(sizeof(double)*3*np), *T = malloc(sizeof(double)*3*np);
    double skin = 0.6;
    build_pairs(&S, skin);
    double maxov = 0, fpist = 0;
    for (int st = 0; st < nsteps; st++) {
        double t = t0 + st*dt;
        double zb = shake_amp*sin(2*M_PI*shake_freq*t);           // base vibrante (opcional)
        double vzb = shake_amp*2*M_PI*shake_freq*cos(2*M_PI*shake_freq*t);
        // rebuild?
        double dmax = 0;
        for (int i = 0; i < np; i++) {
            double dx = pos[3*i]-pos_at_build[3*i], dy = pos[3*i+1]-pos_at_build[3*i+1], dz = pos[3*i+2]-pos_at_build[3*i+2];
            double dd = dx*dx+dy*dy+dz*dz; if (dd > dmax) dmax = dd;
        }
        if (sqrt(dmax) > skin*0.45) build_pairs(&S, skin);
        // posições das esferas
        for (int i = 0; i < np; i++) for (int k = 0; k < sp_n[i]; k++) {
            int s = sp_start[i]+k; double o[3];
            rotate(&q[4*i], &sp_loc[4*s], o);
            spw[4*s] = pos[3*i]+o[0]; spw[4*s+1] = pos[3*i+1]+o[1]; spw[4*s+2] = pos[3*i+2]+o[2]; spw[4*s+3] = sp_loc[4*s+3] + g_coat;
        }
        for (int i = 0; i < 3*np; i++) { F[i] = 0; T[i] = 0; }
        maxov = 0; fpist = 0;
        for (int p = 0; p < npairs; p++) {
            int a = pairs[2*p], b = pairs[2*p+1];
            double dx = pos[3*a]-pos[3*b], dy = pos[3*a+1]-pos[3*b+1], dz = pos[3*a+2]-pos[3*b+2];
            double rc = brad[a]+brad[b]+2*g_coat; if (dx*dx+dy*dy+dz*dz > rc*rc) continue;
            for (int ia = sp_start[a]; ia < sp_start[a]+sp_n[a]; ia++) {
                double *ca = &spw[4*ia];
                double ex = ca[0]-pos[3*b], ey = ca[1]-pos[3*b+1], ez = ca[2]-pos[3*b+2], rr = ca[3]+brad[b]+g_coat;
                if (ex*ex+ey*ey+ez*ez > rr*rr) continue;
                for (int ib = sp_start[b]; ib < sp_start[b]+sp_n[b]; ib++) {
                    double *cb = &spw[4*ib];
                    double fx = ca[0]-cb[0], fy = ca[1]-cb[1], fz = ca[2]-cb[2], r2 = ca[3]+cb[3];
                    if (fx*fx+fy*fy+fz*fz >= r2*r2) continue;
                    contact(F, T, a, b, ca, ca[3], cb, cb[3], &S, kn, zeta, mu, &maxov);
                }
            }
        }
        // paredes
        for (int i = 0; i < np; i++) {
            for (int s = sp_start[i]; s < sp_start[i]+sp_n[i]; s++) {
                double *c = &spw[4*s], r = c[3];
                double rxy = sqrt(c[0]*c[0]+c[1]*c[1]);
                if (rxy + r > R && rxy > 1e-9) {        // parede do molde: centro virtual fora
                    double sc = R/rxy; double cw[3] = {c[0]*sc, c[1]*sc, c[2]};
                    contact(F, T, i, -1, c, r, cw, 0.0 + 0.0, &S, kn, zeta, mu, &maxov);
                    // a esfera virtual tem raio 0 posicionada a R+r: sobreposição = r - (R+r-rxy) = rxy + r - R ✓
                }
                if (c[2] - r < zb) {                    // base
                    double cw[3] = {c[0], c[1], zb - 0.0};
                    // ponto na base com raio 0: dist = c2 - zb ; ov = r - dist
                    double vsave = vel[3*i+2]; vel[3*i+2] -= vzb;
                    contact(F, T, i, -1, c, r, cw, 0.0, &S, kn, zeta, mu, &maxov);
                    vel[3*i+2] = vsave;
                }
                if (c[2] + r > S.zp) {                  // pistão
                    double cw[3] = {c[0], c[1], S.zp};
                    double fz0 = F[3*i+2];
                    double vsave = vel[3*i+2]; vel[3*i+2] += vp*(S.zp > zp_target);
                    contact(F, T, i, -1, c, r, cw, 0.0, &S, kn, zeta, mu, &maxov);
                    vel[3*i+2] = vsave;
                    fpist += -(F[3*i+2]-fz0);
                }
            }
        }
        // integração (semi-implícita) + amortecimento global
        double ke = 0;
        for (int i = 0; i < np; i++) {
            double m = mass[i];
            for (int k = 0; k < 3; k++) {
                double f = F[3*i+k] - gdamp*m*vel[3*i+k];
                if (k == 2) f -= m*g;
                vel[3*i+k] += f/m*dt; pos[3*i+k] += vel[3*i+k]*dt;
                ke += 0.5*m*vel[3*i+k]*vel[3*i+k];
            }
            // rotação: no referencial do corpo
            double tb[3], wb[3]; rotate_inv(&q[4*i], &T[3*i], tb); rotate_inv(&q[4*i], &w[3*i], wb);
            double *I = &Ib[3*i];
            double dw0 = (tb[0] - (I[2]-I[1])*wb[1]*wb[2])/I[0] - gdamp*wb[0];
            double dw1 = (tb[1] - (I[0]-I[2])*wb[2]*wb[0])/I[1] - gdamp*wb[1];
            double dw2 = (tb[2] - (I[1]-I[0])*wb[0]*wb[1])/I[2] - gdamp*wb[2];
            wb[0] += dw0*dt; wb[1] += dw1*dt; wb[2] += dw2*dt;
            rotate(&q[4*i], wb, &w[3*i]);
            double *qq = &q[4*i], *ww = &w[3*i];
            double dq0 = 0.5*(-ww[0]*qq[1]-ww[1]*qq[2]-ww[2]*qq[3]);
            double dq1 = 0.5*( ww[0]*qq[0]+ww[1]*qq[3]-ww[2]*qq[2]);
            double dq2 = 0.5*(-ww[0]*qq[3]+ww[1]*qq[0]+ww[2]*qq[1]);
            double dq3 = 0.5*( ww[0]*qq[2]-ww[1]*qq[1]+ww[2]*qq[0]);
            qq[0]+=dq0*dt; qq[1]+=dq1*dt; qq[2]+=dq2*dt; qq[3]+=dq3*dt;
            double nq = sqrt(qq[0]*qq[0]+qq[1]*qq[1]+qq[2]*qq[2]+qq[3]*qq[3]);
            for (int k = 0; k < 4; k++) qq[k] /= nq;
        }
        if (S.zp > zp_target) { S.zp -= vp*dt; if (S.zp < zp_target) S.zp = zp_target; }
        out[2] = ke;
    }
    double ztop = -1e9;
    for (int s = 0; s < ns; s++) if (spw[4*s+2]+spw[4*s+3] > ztop) ztop = spw[4*s+2]+spw[4*s+3];
    out[0] = maxov; out[1] = fpist; out[3] = ztop; out[4] = S.zp; *zp_io = S.zp;
    free(F); free(T);
    return npairs;
}


In [ ]:
%%writefile md_core.py
"""
Corpo de prova Marshall digital — método de Awuah, Garcia-Hernandez & Valentin (2024),
Construction and Building Materials 416, 135281.

Etapas:
 1. composicao()  -> massas e volumes a partir da granulometria, teor de betume e vazios
 2. agregados()   -> biblioteca de formas (Weibull do AR, Tabela 3) e lista de partículas > 2 mm
 3. simular()     -> despejo no molde e compactação (DEM de aglomerados de esferas, dem.c)
 4. voxelizar()   -> agregados em voxels; mastique e vazios de ar no espaço restante
 5. exportar()    -> STL, pilha TIFF de fases, seções PNG e visualizador HTML autocontido
"""
import json, os, sys, time, pickle, ctypes, subprocess, platform, base64, zipfile
import numpy as np
from scipy import ndimage
from scipy.spatial import ConvexHull

AQUI = os.path.dirname(os.path.abspath(__file__))
EQ3 = {  # Tabela 6 do paper: phi = (b0 + b1 M + b2 NA) / (1 + b3 M + b4 NA), por 250 g
    'trabalhabilidade_pct': [1.70e9, 1.07e7, -9.90e5, 5.90e5, -6.86e3],
    'estabilidade_kN':      [6.25, -1.50e-3, -1.92e-3, -9.52e-3, -2.51e-4],
    'fluencia_mm':          [2.62e-1, 3.97e-2, -5.13e-4, -1.00e-2, -2.62e-4],
    'vazios_pct':           [4.36e8, -2.62e6, -2.0e5, 5.62e5, 2.13e2],
}


def log(*a):
    print(*a, flush=True)


def eq3(M, NA):
    return {k: (b[0]+b[1]*M+b[2]*NA)/(1+b[3]*M+b[4]*NA) for k, b in EQ3.items()}


# ---------------------------------------------------------------- 1. composição
def composicao(cfg):
    s = np.array(cfg['granulometria']['peneiras_mm'], float)
    p = np.array(cfg['granulometria']['passante_pct'], float)
    o = np.argsort(-s); s, p = s[o], p[o]
    Pb = cfg['teor_betume_pct']; ra, rb = cfg['densidade_agregado'], cfg['densidade_betume']
    D = cfg['corpo_de_prova']['diametro_mm']; H = cfg['corpo_de_prova']['altura_mm']; R = D/2
    V = np.pi*R**2*H/1000
    Gmm = 100/((100-Pb)/ra + Pb/rb); Gmb = Gmm*(1-cfg['vazios_alvo_pct']/100)
    mass = Gmb*V; m_agg = mass*(1-Pb/100); m_bit = mass*Pb/100
    p2 = float(np.interp(2.0, s[::-1], p[::-1]))
    m_fine = m_agg*p2/100; m_coarse = m_agg - m_fine
    c = dict(D=D, H=H, R=R, V=V, Gmm=Gmm, Gmb=Gmb, mass=mass, m_agg=m_agg, m_bit=m_bit, pass2=p2,
             m_fine=m_fine, m_coarse=m_coarse, V_coarse=m_coarse/ra,
             M=m_fine/ra + m_bit/rb, V_void=V*cfg['vazios_alvo_pct']/100, k250=250/mass)
    # frações > 2 mm entre peneiras consecutivas
    edges = sorted(set([x for x in s if x > 2] + [2.0]))
    fr = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        pct = np.interp(hi, s[::-1], p[::-1]) - np.interp(lo, s[::-1], p[::-1])
        if pct > 0: fr.append((lo, hi, pct))
    top = s[p >= 99.999].min() if np.any(p >= 99.999) else s.max()
    c['fracoes'] = fr; c['dmax'] = top
    return c


# ---------------------------------------------------------------- 2. agregados
def _forma(rng, par, ang, ngrid=22):
    AR = np.clip(par['AR_escala']*rng.weibull(par['AR_forma']), 0.40, 0.98)
    h = np.clip(par['h']*rng.normal(1, 0.08), 0.42, 0.95)
    dims = np.array([1/AR, 1.0, h])
    npts = 36
    areas = np.array([dims[1]*dims[2]]*2 + [dims[0]*dims[2]]*2 + [dims[0]*dims[1]]*2)
    face = rng.choice(6, npts, p=areas/areas.sum())
    pts = rng.uniform(-0.5, 0.5, (npts, 3))*dims
    ax = face//2; sg = np.where(face % 2 == 0, -1, 1)
    pts[np.arange(npts), ax] = sg*dims[ax]/2*(1-rng.uniform(0, ang, npts))
    v = pts[ConvexHull(pts).vertices]; v = v - v.mean(0)
    v = v*(dims/(v.max(0)-v.min(0)))
    hull = ConvexHull(v); v = v[hull.vertices]; hull = ConvexHull(v); eq = hull.equations
    g = np.stack(np.meshgrid(*[np.linspace(v[:, i].min(), v[:, i].max(), ngrid) for i in range(3)],
                             indexing='ij'), -1).reshape(-1, 3)
    d = (-(g @ eq[:, :3].T + eq[:, 3])).min(1); gi = g[d > 0]
    c = gi.mean(0); vol = hull.volume; r_ = gi - c
    I = (np.eye(3)*np.einsum('ij,ij->i', r_, r_).sum() - r_.T @ r_)/len(gi)*vol
    w, Q = np.linalg.eigh(I)
    v = (v - c) @ Q; gi = (gi - c) @ Q
    hull = ConvexHull(v); eq = hull.equations
    di = (-(gi @ eq[:, :3].T + eq[:, 3])).min(1)
    # aglomerado de esferas: escolha gulosa por volume coberto
    rr = di*1.10
    cidx = rng.choice(len(gi), min(500, len(gi)), replace=False)
    Dm = np.linalg.norm(gi[cidx][:, None, :] - gi[None, :, :], axis=2) < rr[cidx][:, None]
    cov = np.zeros(len(gi), bool); sp = []
    for _ in range(10):
        gain = (Dm & ~cov[None, :]).sum(1)*(rr[cidx] > 0.07)
        j = int(np.argmax(gain))
        if gain[j] < 0.01*len(gi): break
        sp.append((*gi[cidx[j]], rr[cidx[j]])); cov |= Dm[j]
        if cov.mean() > 0.96: break
    f = hull.simplices.copy(); cen = v[f].mean(1)
    n = np.cross(v[f[:, 1]]-v[f[:, 0]], v[f[:, 2]]-v[f[:, 0]])
    flip = (n*cen).sum(1) < 0; f[flip] = f[flip][:, ::-1]
    return dict(verts=v, faces=f, vol=vol, I=w, spheres=np.array(sp))


def agregados(cfg, c, ntpl=80):
    rng = np.random.default_rng(cfg['simulacao']['semente'])
    cls = cfg['forma_agregados']['classes']; ang = cfg['forma_agregados'].get('angulosidade', 0.28)
    T, off = [], []
    for par in cls:
        off.append(len(T)); T += [_forma(rng, par, ang) for _ in range(ntpl)]
    log(f'  {len(T)} formas geradas')

    def classe(d):
        for k, par in enumerate(cls):
            if d <= par['ate_mm'] + 1e-9: return k
        return len(cls)-1

    def sortear(k_, seed=7):
        r = np.random.default_rng(seed); parts = []; rel = {}
        for lo, hi, pct in c['fracoes']:
            alvo = c['m_agg']*pct/100; acc = 0.0; n = 0; kc = classe(hi)
            while True:
                d = np.exp(r.uniform(np.log(lo), np.log(hi))); s = k_*d
                ti = off[kc] + int(r.integers(ntpl))
                mg = T[ti]['vol']*s**3*cfg['densidade_agregado']/1000
                if acc + mg - alvo > alvo - acc: break
                acc += mg; n += 1; parts.append((ti, s))
            rel[f'{lo:g}-{hi:g} mm'] = dict(n=n, massa_g=round(acc, 1), alvo_g=round(alvo, 1))
        return np.array(parts), rel

    cal = cfg['calibracao_tamanho']
    if cal['modo'] == 'NA_250g':
        lo, hi = 0.6, 2.0
        for _ in range(28):
            km = (lo+hi)/2
            if len(sortear(km)[0])*c['k250'] > cal['NA_250g']: lo = km
            else: hi = km
        k_ = (lo+hi)/2
    else:
        k_ = cal['fator']
    parts, rel = sortear(k_)
    log(f'  fator Min Feret/peneira = {k_:.3f}; NA = {len(parts)} ({len(parts)*c["k250"]:.0f} por 250 g)')
    return T, parts, rel, k_


# ---------------------------------------------------------------- 3. simulação
def compilar_dem():
    src = os.path.join(AQUI, 'dem.c')
    ext = '.dll' if platform.system() == 'Windows' else '.so'
    lib = os.path.join(AQUI, 'libdem' + ext)
    if os.path.exists(lib) and os.path.getmtime(lib) > os.path.getmtime(src):
        return lib
    win = platform.system() == 'Windows'
    cmds = [[cc, '-O3', '-shared', src, '-o', lib] + ([] if win else ['-fPIC', '-lm'])
            for cc in (['gcc', 'clang'] if win else ['cc', 'gcc', 'clang'])]
    zig = [sys.executable, '-m', 'ziglang', 'cc', '-O3', '-shared', src, '-o', lib] + ([] if win else ['-fPIC', '-lm'])
    cmds.append(zig)
    for cmd in cmds:
        try:
            subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return lib
        except (OSError, subprocess.CalledProcessError):
            continue
    # último recurso: instala o compilador zig pelo pip e tenta de novo
    try:
        log('  instalando compilador C (pip install ziglang)…')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ziglang'])
        subprocess.check_call(zig)
        return lib
    except (OSError, subprocess.CalledProcessError):
        pass
    sys.exit('Não encontrei um compilador C (gcc/clang). Use o notebook no Google Colab, '
             'ou instale: macOS -> "xcode-select --install"; Windows -> WSL ou MinGW-w64.')


def simular(cfg, c, T, parts, pasta):
    lib = ctypes.CDLL(compilar_dem())
    Dp = np.ctypeslib.ndpointer(np.float64, flags='C'); Ip = np.ctypeslib.ndpointer(np.int32, flags='C')
    dbl = ctypes.c_double
    lib.run.argtypes = [ctypes.c_int, Ip, Ip, Dp, Dp, Dp, Dp, Dp, Dp, Dp, Dp, dbl, ctypes.POINTER(dbl),
                        dbl, dbl, ctypes.c_int] + [dbl]*9 + [Dp]
    lib.run.restype = ctypes.c_int
    lib.set_coat.argtypes = [dbl]
    rho = cfg['densidade_agregado']*1e-6
    R, H = c['R'], c['H']
    rng = np.random.default_rng(cfg['simulacao']['semente'] + 1)
    parts = parts[rng.permutation(len(parts))]
    n = len(parts)
    st, sn, sl = [], [], []; mass = np.zeros(n); Ib = np.zeros((n, 3)); brad = np.zeros(n); k = 0
    for i, (ti, s) in enumerate(parts):
        t = T[int(ti)]; sp = t['spheres']*s
        st.append(k); sn.append(len(sp)); k += len(sp); sl.append(sp)
        mass[i] = t['vol']*s**3*rho; Ib[i] = t['I']*s**5*rho
        brad[i] = (np.linalg.norm(sp[:, :3], axis=1) + sp[:, 3]).max()
    st = np.array(st, np.int32); sn = np.array(sn, np.int32); sl = np.ascontiguousarray(np.vstack(sl))
    # chuva inicial sem sobreposição
    pos = np.zeros((n, 3)); z = 0.0
    for i in range(n):
        rb = brad[i]
        for tries in range(200):
            rr = (R - rb - 0.2)*np.sqrt(rng.uniform()); th = rng.uniform(0, 2*np.pi)
            p = np.array([rr*np.cos(th), rr*np.sin(th), z + rb + rng.uniform(0, 2)])
            P = pos[max(0, i-500):i]; B = brad[max(0, i-500):i]
            if i and np.any(np.linalg.norm(P - p, axis=1) < B + rb + 0.1):
                if tries % 20 == 19: z += 0.5
                continue
            break
        pos[i] = p
    q = rng.normal(size=(n, 4)); q /= np.linalg.norm(q, axis=1)[:, None]
    pos = np.ascontiguousarray(pos); q = np.ascontiguousarray(q)
    vel = np.zeros((n, 3)); w = np.zeros((n, 3)); Ib = np.ascontiguousarray(Ib)
    zp = dbl(1e4); out = np.zeros(5); tt = [0.0]; dt = 1e-5

    def run(ns, mu, gd, zpt=1e4, vp=0.0):
        lib.run(n, st, sn, sl, pos, q, vel, w, mass, Ib, brad, R, ctypes.byref(zp), zpt, vp,
                ns, dt, 9810., 1e3, 0.3, mu, gd, 0., 0., tt[0], out)
        tt[0] += ns*dt

    t0 = time.time(); sim = cfg['simulacao']
    lib.set_coat(0.0)
    log(f'  despejo de {n} agregados (coluna inicial de {pos[:, 2].max():.0f} mm)…')
    for it in range(600):
        run(500, sim['atrito_despejo'], 3.0)
        if it % 20 == 0:
            log(f'    t={tt[0]:.2f}s topo={out[3]:.1f} mm ({time.time()-t0:.0f}s)')
        if tt[0] > 0.3 and out[2] < 50: break
    log(f'  leito após despejo: {out[3]:.1f} mm; compactando até {H} mm…')
    zp.value = max(out[3], H) + 0.5
    while zp.value > H + 1e-9:
        run(500, 0.15, 4.0, H, 80.0)
    coat = 0.0; Fstop = sim['forca_final_pistao_N']*1e3
    while out[1] <= Fstop and coat < 1.0:
        coat += 0.004; lib.set_coat(coat)
        for _ in range(4): run(250, sim['atrito_compactacao'], 6.0, H, 0.0)
    log(f'  película de mastique equivalente: {coat:.3f} mm; força no pistão {out[1]/1e3:.1f} N')
    for _ in range(30): run(500, 0.35, 8.0, H, 0.0)
    log(f'  simulação concluída em {time.time()-t0:.0f} s')
    np.savez(os.path.join(pasta, 'estado_final.npz'), pos=pos, q=q, parts=parts, coat=coat)
    return pos, q, parts, coat


# ---------------------------------------------------------------- 4. voxelização
def _qmat(q):
    w, a, b, c = q
    return np.array([[1-2*(b*b+c*c), 2*(a*b-w*c), 2*(a*c+w*b)],
                     [2*(a*b+w*c), 1-2*(a*a+c*c), 2*(b*c-w*a)],
                     [2*(a*c-w*b), 2*(b*c+w*a), 1-2*(a*a+b*b)]])


def voxelizar(cfg, c, T, pos, q, parts):
    h = cfg['voxel_mm']; R, H = c['R'], c['H']
    nx = int(np.ceil(2*R/h)); nz = int(np.ceil(H/h))
    x = (np.arange(nx)+0.5)*h - R; z = (np.arange(nz)+0.5)*h
    lab = np.zeros((nx, nx, nz), np.int32); over = 0
    eqs = [ConvexHull(t['verts']).equations for t in T]
    for i, (ti, s) in enumerate(parts):
        t = T[int(ti)]; Rm = _qmat(q[i]); V = t['verts']*s @ Rm.T + pos[i]
        lo = np.maximum(np.floor((V.min(0) - [-R, -R, 0])/h).astype(int), 0)
        hi = np.minimum(np.ceil((V.max(0) - [-R, -R, 0])/h).astype(int), [nx, nx, nz])
        if np.any(hi <= lo): continue
        X, Y, Z = np.meshgrid(x[lo[0]:hi[0]], x[lo[1]:hi[1]], z[lo[2]:hi[2]], indexing='ij')
        P = np.stack([X, Y, Z], -1).reshape(-1, 3) - pos[i]
        e = eqs[int(ti)]
        ins = ((P @ (e[:, :3] @ Rm.T).T + e[:, 3]*s) <= 0).all(1).reshape(X.shape)
        sub = lab[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
        over += np.count_nonzero(ins & (sub > 0)); sub[ins & (sub == 0)] = i+1
    cyl = (x[:, None]**2 + x[None, :]**2 <= R**2)[:, :, None] & np.ones(nz, bool)[None, None, :]
    lab[~cyl] = 0
    vv = h**3/1000; free = cyl & (lab == 0)
    dist = ndimage.gaussian_filter(ndimage.distance_transform_edt(lab == 0, sampling=h).astype(np.float32), 1.0)
    rng = np.random.default_rng(cfg['simulacao']['semente'] + 2)
    noise = ndimage.gaussian_filter(rng.normal(size=lab.shape).astype(np.float32), 4.0); noise /= noise.std()
    rr = np.sqrt(x[:, None]**2 + x[None, :]**2)[:, :, None]
    dwall = np.minimum(np.minimum(R - rr, z[None, None, :]), H - z[None, None, :]).astype(np.float32)
    score = dist + 0.25*noise + 0.35*np.exp(-dwall/1.0); score[~free] = -1e9
    del noise, dwall, dist
    nvoid = int(round(c['V_void']/vv)); ball = ndimage.generate_binary_structure(3, 1)

    def voids_at(thr):
        v = ndimage.binary_opening(free & (score >= thr), ball)
        lb, _ = ndimage.label(v); sz = np.bincount(lb.ravel()); keep = sz >= 30; keep[0] = False
        return keep[lb]

    lo, hi = np.percentile(score[free], 50), np.percentile(score[free], 99.95)
    void = None
    for _ in range(16):
        thr = (lo+hi)/2; void = voids_at(thr); nv = np.count_nonzero(void)
        if nv > nvoid: lo = thr
        else: hi = thr
        if abs(nv - nvoid) < 0.005*nvoid: break
    phase = np.zeros(lab.shape, np.uint8)
    phase[free & ~void] = 1; phase[void] = 2; phase[lab > 0] = 3
    st = dict(V_cil=np.count_nonzero(cyl)*vv, V_agregados=np.count_nonzero(lab)*vv, V_sobreposicao=over*vv,
              V_mastique=np.count_nonzero(phase == 1)*vv, V_vazios=np.count_nonzero(void)*vv)
    st['AVC'] = st['V_vazios']/st['V_cil']*100
    return phase, lab, st


# ---------------------------------------------------------------- 5. exportação
def _stl(fn, tris, name):
    tris = tris.astype(np.float32)
    nrm = np.cross(tris[:, 1]-tris[:, 0], tris[:, 2]-tris[:, 0])
    nrm /= np.linalg.norm(nrm, axis=1)[:, None] + 1e-12
    rec = np.zeros(len(tris), dtype=[('n', '<f4', 3), ('v', '<f4', (3, 3)), ('a', '<u2')])
    rec['n'] = nrm; rec['v'] = tris
    with open(fn, 'wb') as f:
        f.write(name.encode('ascii', 'replace')[:80].ljust(80, b' '))
        f.write(np.uint32(len(tris)).tobytes()); f.write(rec.tobytes())


def _mc(mask, step, sigma, h, R):
    from skimage.measure import marching_cubes
    m = mask.astype(np.float32)
    if step > 1:
        sh = (np.array(m.shape)//step)*step
        m = m[:sh[0], :sh[1], :sh[2]].reshape(sh[0]//step, step, sh[1]//step, step, sh[2]//step, step).mean((1, 3, 5))
    if sigma: m = ndimage.gaussian_filter(m, sigma)
    v, f, _, _ = marching_cubes(np.pad(m, 1), 0.5)
    v = (v - 0.5)*h*step; v[:, 0] -= R; v[:, 1] -= R
    return v, f


def exportar(cfg, c, T, pos, q, parts, k_, phase, lab, st, coat, pasta):
    from PIL import Image
    h = cfg['voxel_mm']; R, H = c['R'], c['H']; nome = cfg['nome']
    ds = os.path.join(pasta, 'stl'); os.makedirs(ds, exist_ok=True)
    nx, _, nz = phase.shape
    # agregados
    tris = np.concatenate([(T[int(ti)]['verts']*s @ _qmat(q[i]).T + pos[i])[T[int(ti)]['faces']]
                           for i, (ti, s) in enumerate(parts)])
    _stl(os.path.join(ds, f'{nome}_agregados.stl'), tris, f'{nome} agregados')
    v, f = _mc(phase == 2, 2, 0.5, h, R); _stl(os.path.join(ds, f'{nome}_vazios.stl'), v[f], 'vazios')
    v, f = _mc(phase == 1, 4, 0.0, h, R); _stl(os.path.join(ds, f'{nome}_mastique_1mm.stl'), v[f], 'mastique')
    th = np.linspace(0, 2*np.pi, 129)[:-1]
    r0 = np.c_[R*np.cos(th), R*np.sin(th), np.zeros_like(th)]; r1 = r0 + [0, 0, H]; env = []
    for k in range(len(th)):
        a, b = k, (k+1) % len(th)
        env += [[r0[a], r0[b], r1[b]], [r0[a], r1[b], r1[a]], [[0, 0, 0], r0[b], r0[a]], [[0, 0, H], r1[a], r1[b]]]
    _stl(os.path.join(ds, f'{nome}_envelope.stl'), np.array(env), 'envelope')
    pages = [Image.fromarray(phase[:, :, k].T[::-1].copy()) for k in range(nz)]
    pages[0].save(os.path.join(ds, f'{nome}_fases_voxel.tif'), save_all=True, append_images=pages[1:],
                  compression='tiff_deflate')
    # seções PNG (em memória, entram no HTML)
    stone = [(184, 178, 166), (164, 158, 147), (199, 192, 178), (146, 141, 132), (210, 203, 190)]
    fraccol = [(232, 197, 71), (214, 135, 64), (91, 141, 196), (64, 92, 168), (176, 72, 140)]
    pal = [(0, 0, 0), (30, 29, 28), (226, 61, 40)] + stone + fraccol
    palf = sum([list(x) for x in pal], []) + [0]*(768 - 3*len(pal))
    sieve = parts[:, 1]/k_
    edges = [2] + [hi for lo, hi, _ in c['fracoes']]
    frac = np.clip(np.searchsorted(edges, sieve) - 1, 0, 4)
    # rótulos das frações para a legenda (até 5 grupos)
    tone = (np.arange(len(parts))*2654435761 % 5).astype(np.uint8)
    luts = {'s': np.r_[0, 3+tone].astype(np.uint8), 'f': np.r_[0, 8+frac].astype(np.uint8)}
    import io
    sec = {}

    def img(ph, lb, lut):
        a = np.zeros(ph.shape, np.uint8); a[ph == 1] = 1; a[ph == 2] = 2; m = ph == 3; a[m] = lut[lb[m]]
        im = Image.fromarray(a, 'P'); im.putpalette(palf); b = io.BytesIO()
        im.save(b, 'PNG', optimize=True, transparency=0)
        return 'data:image/png;base64,' + base64.b64encode(b.getvalue()).decode()

    zs = np.arange(0.5, H, 1.0); xs = np.arange(-np.floor(R/2)*2, np.floor(R/2)*2+0.1, 2.0)
    for md, lut in luts.items():
        for k, zz in enumerate(zs):
            iz = min(int(zz/h), nz-1); sec[f'{md}z{k}'] = img(phase[:, :, iz].T[::-1], lab[:, :, iz].T[::-1], lut)
        for k, xx in enumerate(xs):
            ix = min(max(int((xx+R)/h), 0), nx-1); sec[f'{md}x{k}'] = img(phase[ix].T[::-1], lab[ix].T[::-1], lut)
    vv, vf = _mc(phase == 2, 3, 0.45, h, R)
    b64 = lambda a: base64.b64encode(np.ascontiguousarray(a).tobytes()).decode()
    tpl = [dict(v=b64(t['verts'].astype(np.float32)), f=b64(t['faces'].astype(np.uint16))) for t in T]
    inst = np.c_[parts[:, 0], parts[:, 1], pos, q, frac, tone].astype(np.float32)
    # textos do painel
    k250 = c['k250']; NA = len(parts); M250 = c['M']*k250; NA250 = NA*k250
    pr = eq3(M250, NA250)
    fmt = lambda x, d=1: f'{x:,.{d}f}'.replace(',', ' ').replace('.', ',')
    nomes = [('trabalhabilidade_pct', 'Trabalhabilidade ω, %'), ('estabilidade_kN', 'Estabilidade S, kN'),
             ('fluencia_mm', 'Fluência F, mm'), ('vazios_pct', 'Vazios AVC, %')]
    head = ['', 'Eq. (3)']; cols = [pr]
    for key, lbl in [('propriedades_nomograma', 'Nomograma'), ('propriedades_medidas', 'Medido')]:
        if cfg.get(key): head.append(lbl); cols.append(cfg[key])
    props = [[lbl] + [fmt(cc[k]) if cc.get(k) is not None else '—' for cc in cols] for k, lbl in nomes]
    props.append(['Vazios no modelo virtual, %', fmt(st['AVC'])] + ['']*(len(head)-2))
    meta = dict(
        title_html=f'Corpo de Prova <span>{nome}</span>',
        spec=f"{cfg.get('descricao', '')} · betume {fmt(cfg['teor_betume_pct'])} % · Marshall Ø{fmt(c['D'])} × {fmt(c['H'])} mm",
        comp=[['Massa total', f"{fmt(c['mass'], 0)} g"], [f"Betume ({fmt(cfg['teor_betume_pct'])} %)", f"{fmt(c['m_bit'])} g"],
              ['Agregados &gt; 2 mm', f"{fmt(c['m_coarse'], 0)} g · {fmt(st['V_agregados'], 0)} cm³"],
              ['Número de agregados NA', f"{fmt(NA, 0)} ({fmt(NA250, 0)})"],
              ['Volume de mastique M', f"{fmt(st['V_mastique'])} cm³ ({fmt(M250)})"],
              ['Vazios de ar', f"{fmt(st['V_vazios'])} cm³"],
              ['Película de mastique', f"{fmt(coat, 2)} mm"],
              ['Gmm · Gmb', f"{fmt(c['Gmm'], 3)} · {fmt(c['Gmb'], 3)} Mg/m³"]],
        props_head=head, props=props,
        propnote=f'Eq. (3) com os coeficientes da Tabela 6 para M = {fmt(M250)} cm³ e NA = {fmt(NA250, 0)} (por 250 g).',
        M250=round(M250, 1), NA250=int(round(NA250)))
    data = dict(R=R, H=H, zs=zs.tolist(), xs=xs.tolist(), tpl=tpl, inst=b64(inst), ninst=NA,
                void_v=b64(vv.astype(np.float32)), void_f=b64(vf.astype(np.uint32)), meta=meta)
    html = open(os.path.join(AQUI, 'viewer_template.html'), encoding='utf-8').read()
    html = html.replace('/*__TITLE__*/', f'Corpo de Prova {nome}')
    html = html.replace('/*__DATA__*/', 'window.V5DATA=' + json.dumps(data) + ';window.V5SEC=' + json.dumps(sec) + ';')
    fv = os.path.join(pasta, f'{nome}_visualizador.html')
    open(fv, 'w', encoding='utf-8').write(html)
    rel = dict(composicao={k: v for k, v in c.items() if k != 'fracoes'}, voxel=st, NA=NA, NA_250g=NA250,
               M_250g=M250, pelicula_mastique_mm=coat, fator_minferet=k_, eq3=pr)
    json.dump(rel, open(os.path.join(pasta, f'{nome}_resultados.json'), 'w'), indent=1, ensure_ascii=False,
              default=float)
    with zipfile.ZipFile(os.path.join(pasta, f'{nome}_STL.zip'), 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in sorted(os.listdir(ds)): zf.write(os.path.join(ds, fn), fn)
    return fv, rel


# ---------------------------------------------------------------- orquestração
def rodar(cfg_path, saida=None):
    cfg = json.load(open(cfg_path, encoding='utf-8'))
    pasta = saida or os.path.join(os.path.dirname(os.path.abspath(cfg_path)), 'resultado_' + cfg['nome'])
    os.makedirs(pasta, exist_ok=True); t0 = time.time()
    log('1/5 composição'); c = composicao(cfg)
    log(f"  massa {c['mass']:.0f} g; agregados > 2 mm {c['V_coarse']:.1f} cm³; mastique {c['M']:.1f} cm³; vazios {c['V_void']:.1f} cm³")
    log('2/5 agregados virtuais'); T, parts, rel, k_ = agregados(cfg, c)
    for k, v in rel.items(): log(f'    {k}: {v["n"]} partículas, {v["massa_g"]} g')
    log('3/5 despejo e compactação'); pos, q, parts, coat = simular(cfg, c, T, parts, pasta)
    log('4/5 mastique e vazios'); phase, lab, st = voxelizar(cfg, c, T, pos, q, parts)
    log(f"  agregados {st['V_agregados']:.1f} cm³, mastique {st['V_mastique']:.1f} cm³, vazios {st['V_vazios']:.1f} cm³ (AVC {st['AVC']:.2f} %)")
    log('5/5 exportação'); fv, res = exportar(cfg, c, T, pos, q, parts, k_, phase, lab, st, coat, pasta)
    log(f'Pronto em {time.time()-t0:.0f} s. Visualizador: {fv}')
    log('Eq. (3): ' + ', '.join(f'{k} = {v:.2f}' for k, v in res['eq3'].items()))
    return pasta


In [ ]:
%%writefile viewer_template.html
<title>/*__TITLE__*/</title>
<meta name="claude:viewport-fit-sides" content="cover">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Archivo:wght@500;700;800&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&display=swap">
<style>
/* Layout: bancada de laboratório — visor 3D à esquerda, ficha técnica do corpo de prova à direita */
:root{
  --bg:#eeece8; --panel:#f8f7f4; --ink:#1f1e1c; --muted:#6b675f; --line:#d6d2ca;
  --accent:#b3611b; --void:#e23d28; --mastic:#1e1d1c; --stage:#d9d6d0;
  --f-display:"Archivo", "Helvetica Neue", Arial, sans-serif;
  --f-body:"IBM Plex Sans", "Segoe UI", Arial, sans-serif;
  --f-mono:"IBM Plex Mono", ui-monospace, Menlo, monospace;
}
@media (prefers-color-scheme: dark){:root:not([data-theme="light"]){
  --bg:#171615; --panel:#201f1d; --ink:#ece9e3; --muted:#a39e94; --line:#36332f;
  --accent:#e0913f; --void:#ff5a3c; --mastic:#0e0e0d; --stage:#2a2826; color-scheme:dark}}
:root[data-theme="dark"]{
  --bg:#171615; --panel:#201f1d; --ink:#ece9e3; --muted:#a39e94; --line:#36332f;
  --accent:#e0913f; --void:#ff5a3c; --mastic:#0e0e0d; --stage:#2a2826; color-scheme:dark}
html,body{height:100%}
body{background:var(--bg);color:var(--ink);font-family:var(--f-body);font-size:14px;line-height:1.5}
.wrap{padding-left:max(16px,env(safe-area-inset-left,0px));padding-right:max(16px,env(safe-area-inset-right,0px));padding-block:16px 24px;max-width:1440px;margin:0 auto;display:flex;flex-direction:column;gap:14px}
header{display:flex;flex-wrap:wrap;align-items:baseline;justify-content:space-between;gap:6px 20px;border-bottom:2px solid var(--ink);padding-bottom:10px}
h1{font-family:var(--f-display);font-weight:800;font-size:clamp(24px,3.2vw,34px);letter-spacing:-.01em;margin:0;line-height:1.05;text-wrap:balance}
h1 span{color:var(--accent)}
.spec{font-family:var(--f-mono);font-size:12px;color:var(--muted)}
.grid{display:grid;grid-template-columns:minmax(0,1fr) 360px;gap:16px;align-items:start}
@media (max-width:960px){.grid{grid-template-columns:minmax(0,1fr)}}
.stage{position:relative;background:var(--stage);border:1px solid var(--line);border-radius:6px;overflow:hidden;height:min(72vh,720px);min-height:420px}
#view3d,#view2d{position:absolute;inset:0}
#view2d{display:flex;align-items:center;justify-content:center;padding:24px}
#view2d img{max-width:100%;max-height:100%;image-rendering:auto;filter:drop-shadow(0 2px 8px rgba(0,0,0,.25))}
.tabs{display:flex;gap:4px;position:absolute;top:10px;left:10px;z-index:3}
.tabs button,.seg button{font:500 12px var(--f-body);border:1px solid var(--line);background:var(--panel);color:var(--ink);padding:6px 10px;border-radius:4px;cursor:pointer}
.tabs button[aria-pressed="true"],.seg button[aria-pressed="true"]{background:var(--ink);color:var(--bg);border-color:var(--ink)}
button:focus-visible,input:focus-visible{outline:2px solid var(--accent);outline-offset:2px}
.hud{position:absolute;bottom:10px;left:10px;right:10px;z-index:3;display:flex;flex-wrap:wrap;gap:8px 14px;align-items:center;background:color-mix(in srgb,var(--panel) 90%,transparent);border:1px solid var(--line);border-radius:6px;padding:8px 10px;font-size:12px}
.hud label{display:flex;align-items:center;gap:6px;color:var(--muted)}
.hud input[type=range]{width:min(220px,40vw);accent-color:var(--accent)}
.seg{display:flex;gap:2px}
.mono{font-family:var(--f-mono);font-variant-numeric:tabular-nums}
.loading{position:absolute;inset:0;display:flex;align-items:center;justify-content:center;font-family:var(--f-mono);color:var(--muted);z-index:2}
aside{display:flex;flex-direction:column;gap:14px;min-width:0}
.card{background:var(--panel);border:1px solid var(--line);border-radius:6px;padding:14px 16px}
.card h2{font-family:var(--f-display);font-weight:700;font-size:13px;letter-spacing:.08em;text-transform:uppercase;margin:0 0 10px;color:var(--muted)}
table{width:100%;border-collapse:collapse;font-size:13px}
td,th{padding:4px 0;border-bottom:1px solid var(--line);text-align:right;font-variant-numeric:tabular-nums}
td:first-child,th:first-child{text-align:left;color:var(--muted);font-weight:400}
th{font-weight:500;font-size:11px;color:var(--muted);letter-spacing:.04em}
tr:last-child td{border-bottom:none}
.legend{display:flex;flex-wrap:wrap;gap:6px 14px;font-size:12px}
.legend i{display:inline-block;width:12px;height:12px;border-radius:2px;vertical-align:-2px;margin-right:5px;border:1px solid rgba(0,0,0,.2)}
.note{font-size:12px;color:var(--muted);margin:8px 0 0}
.calc{display:grid;grid-template-columns:auto 1fr auto;gap:6px 10px;align-items:center;font-size:12px}
.calc input{accent-color:var(--accent);width:100%}
.out4{display:grid;grid-template-columns:repeat(4,1fr);gap:6px;margin-top:10px}
.out4 div{border:1px solid var(--line);border-radius:4px;padding:6px;text-align:center}
.out4 b{display:block;font-family:var(--f-mono);font-size:15px}
.out4 span{font-size:11px;color:var(--muted)}
.ok{color:#2f7d4b}.bad{color:var(--void)}
@media (prefers-reduced-motion:reduce){*{transition:none!important}}
</style>

<div class="wrap">
  <header>
    <h1 id="ttl">Corpo de Prova</h1>
    <div class="spec" id="spec"></div>
  </header>
  <div class="grid">
    <div class="stage" id="stage">
      <div class="tabs" role="group" aria-label="Vista">
        <button id="t3d" aria-pressed="true">Modelo 3D</button>
        <button id="t2d" aria-pressed="false">Seções (tomografia)</button>
      </div>
      <div id="view3d"></div>
      <div id="view2d" hidden><img id="secimg" alt="Seção do corpo de prova"></div>
      <div class="loading" id="loading">carregando o modelo…</div>
      <div class="hud" id="hud">
        <div class="seg" role="group" aria-label="Exibição" id="segShow">
          <button data-v="real" aria-pressed="true">Com mastique</button>
          <button data-v="agg" aria-pressed="false">Só agregados</button>
        </div>
        <div class="seg" role="group" aria-label="Cor" id="segColor">
          <button data-v="s" aria-pressed="true">Calcário</button>
          <button data-v="f" aria-pressed="false">Por fração</button>
        </div>
        <div class="seg" role="group" aria-label="Corte" id="segCut">
          <button data-v="none" aria-pressed="false">Sem corte</button>
          <button data-v="x" aria-pressed="true">Corte vertical</button>
          <button data-v="z" aria-pressed="false">Corte horizontal</button>
        </div>
        <label for="cutpos">posição <input type="range" id="cutpos" min="0" max="100" value="50"> <span class="mono" id="cutlbl">x = 0 mm</span></label>
        <label for="voids"><input type="checkbox" id="voids" checked> vazios</label>
      </div>
    </div>

    <aside>
      <div class="card">
        <h2>Legenda</h2>
        <div class="legend" id="legend"></div>
      </div>
      <div class="card">
        <h2>Composição (corpo de prova completo)</h2>
        <table id="comp"></table>
        <p class="note">Mastique = betume + fíler + agregados com Min Feret &lt; 2 mm, como no paper. Valores por 250 g entre parênteses, a escala usada pelo método.</p>
      </div>
      <div class="card">
        <h2>Propriedades</h2>
        <table id="props"></table>
        <p class="note" id="propnote"></p>
      </div>
      <div class="card">
        <h2>Equação (3) — explorar</h2>
        <div class="calc">
          <label for="cM">Mastique M</label><input type="range" id="cM" min="25" max="50" step="0.1" value="37.3" data-k="M"><span class="mono" id="cMv"></span>
          <label for="cN">Agregados NA</label><input type="range" id="cN" min="900" max="1500" step="1" value="1196" data-k="N"><span class="mono" id="cNv"></span>
        </div>
        <div class="out4" id="calcout"></div>
        <p class="note">Por 250 g de mistura. Verde = dentro dos limites da Tabela 7 (ω 50–80 %, S &gt; 6,7 kN, F 2–4 mm, AVC 3–5 %).</p>
      </div>
    </aside>
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>/*__DATA__*/</script>
<script>
if (!window.THREE) { document.getElementById("loading").textContent = "Não foi possível carregar a biblioteca 3D (three.js)."; }
else {
try {

function OrbitControls(cam, el){
  const self = this; this.target = new THREE.Vector3(); this.enableDamping = false; const L = [];
  let r = 0, th = 0, ph = 0, drag = null;
  const sync = () => { const o = cam.position.clone().sub(self.target); r = o.length(); th = Math.atan2(o.x, o.z); ph = Math.acos(Math.min(1, Math.max(-1, o.y/r))); };
  const put = () => { cam.position.set(self.target.x + r*Math.sin(ph)*Math.sin(th), self.target.y + r*Math.cos(ph), self.target.z + r*Math.sin(ph)*Math.cos(th)); cam.lookAt(self.target); L.forEach(f=>f()); };
  this.addEventListener = (n, f) => L.push(f);
  this.update = () => { if (r === 0) { sync(); put(); } return false; };
  el.style.touchAction = 'none';
  const pts = new Map(); let pinch = 0;
  el.addEventListener('pointerdown', e => { el.setPointerCapture(e.pointerId); pts.set(e.pointerId, [e.clientX, e.clientY]); drag = {x:e.clientX, y:e.clientY, pan: e.button===2 || e.shiftKey}; sync(); });
  el.addEventListener('pointermove', e => { if (!pts.has(e.pointerId)) return; pts.set(e.pointerId, [e.clientX, e.clientY]);
    if (pts.size === 2) { const [a, b] = [...pts.values()], d = Math.hypot(a[0]-b[0], a[1]-b[1]); if (pinch) { r = Math.min(800, Math.max(60, r*pinch/d)); put(); } pinch = d; return; }
    const dx = e.clientX - drag.x, dy = e.clientY - drag.y; drag.x = e.clientX; drag.y = e.clientY;
    if (drag.pan) { const k = r/700, right = new THREE.Vector3().setFromMatrixColumn(cam.matrix, 0), up = new THREE.Vector3().setFromMatrixColumn(cam.matrix, 1);
      self.target.addScaledVector(right, -dx*k).addScaledVector(up, dy*k); }
    else { th -= dx*0.008; ph = Math.min(3.1, Math.max(0.05, ph - dy*0.008)); }
    put(); });
  const up = e => { pts.delete(e.pointerId); if (pts.size < 2) pinch = 0; };
  el.addEventListener('pointerup', up); el.addEventListener('pointercancel', up);
  el.addEventListener('contextmenu', e => e.preventDefault());
  el.addEventListener('wheel', e => { e.preventDefault(); sync(); r = Math.min(800, Math.max(60, r*Math.exp(e.deltaY*0.001))); put(); }, {passive:false});
}


const D = window.V5DATA;
const R = D.R, H = D.H;
const b64f = (s, T) => { const b = atob(s), u = new Uint8Array(b.length); for (let i = 0; i < b.length; i++) u[i] = b.charCodeAt(i); return new T(u.buffer); };

const STONE = [[184,178,166],[164,158,147],[199,192,178],[146,141,132],[210,203,190]];
const FRAC  = [[232,197,71],[214,135,64],[91,141,196],[64,92,168],[176,72,140]];
const FRACL = ['2–2,8 mm','2,8–4 mm','4–6,3 mm','6,3–8 mm','8–10 mm'];
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();

// ---------- painel: composição ----------
const MT = D.meta;
document.getElementById('ttl').innerHTML = MT.title_html; document.getElementById('spec').textContent = MT.spec;
document.getElementById('comp').innerHTML = MT.comp.map(r => `<tr><td>${r[0]}</td><td class="mono">${r[1]}</td></tr>`).join('');
document.getElementById('props').innerHTML = '<tr>' + MT.props_head.map(h=>`<th>${h}</th>`).join('') + '</tr>' +
  MT.props.map(r => '<tr>' + r.map((c,i)=> i ? `<td class="mono">${c}</td>` : `<td>${c}</td>`).join('') + '</tr>').join('');
document.getElementById('propnote').textContent = MT.propnote;

// ---------- calculadora Eq. (3) ----------
const B = {w:[1.70e9,1.07e7,-9.90e5,5.90e5,-6.86e3], S:[6.25,-1.50e-3,-1.92e-3,-9.52e-3,-2.51e-4], F:[2.62e-1,3.97e-2,-5.13e-4,-1.00e-2,-2.62e-4], AVC:[4.36e8,-2.62e6,-2.0e5,5.62e5,2.13e2]};
const eq3 = (b, M, N) => (b[0]+b[1]*M+b[2]*N)/(1+b[3]*M+b[4]*N);
const LIM = {w:[50,80], S:[6.7,1e9], F:[2,4], AVC:[3,5]};
const LBL = {w:'ω %', S:'S kN', F:'F mm', AVC:'AVC %'};
function calc(){
  const M = +cM.value, N = +cN.value;
  cMv.textContent = M.toFixed(1)+' cm³'; cNv.textContent = N;
  calcout.innerHTML = Object.keys(B).map(k => { const v = eq3(B[k], M, N), ok = v >= LIM[k][0] && v <= LIM[k][1];
    return `<div><b class="${ok?'ok':'bad'}">${v.toFixed(1).replace('.',',')}</b><span>${LBL[k]}</span></div>`; }).join('');
}
const cM = document.getElementById('cM'), cN = document.getElementById('cN'), cMv = document.getElementById('cMv'), cNv = document.getElementById('cNv'), calcout = document.getElementById('calcout');
cM.value = MT.M250; cN.value = MT.NA250; cM.oninput = cN.oninput = calc; calc();


// ---------- estado ----------
const st = { show:'real', color:'s', cut:'x', pos:50, voids:true, tab:'3d' };
const legend = document.getElementById('legend');
function drawLegend(){
  const items = [['Mastique', css('--mastic')], ['Vazios de ar', css('--void')]];
  if (st.color === 's') items.push(['Agregado (calcário)', `rgb(${STONE[0]})`]);
  else FRAC.forEach((c,i)=>items.push([FRACL[i], `rgb(${c})`]));
  legend.innerHTML = items.map(([l,c])=>`<span><i style="background:${c}"></i>${l}</span>`).join('');
}

// ---------- 3D ----------
const host = document.getElementById('view3d');
const renderer = new THREE.WebGLRenderer({antialias:true, alpha:true});
renderer.setPixelRatio(Math.min(devicePixelRatio, 2));
renderer.localClippingEnabled = true; renderer.outputEncoding = THREE.sRGBEncoding;
host.appendChild(renderer.domElement);
const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(32, 1, 1, 2000);
camera.position.set(150, 120, 190);
const controls = new OrbitControls(camera, renderer.domElement);
controls.target.set(0, H/2, 0); controls.enableDamping = true;
scene.add(new THREE.HemisphereLight(0xffffff, 0x4a443c, 0.75));
const sun = new THREE.DirectionalLight(0xffffff, 0.9); sun.position.set(120, 220, 140); scene.add(sun);
const fill = new THREE.DirectionalLight(0xffffff, 0.3); fill.position.set(-150, 60, -100); scene.add(fill);

const model = new THREE.Group(); model.rotation.x = -Math.PI/2; scene.add(model);   // modelo em z-para-cima
const clip = new THREE.Plane(new THREE.Vector3(-1,0,0), 0);
const planes = [clip];

// agregados: um InstancedMesh por forma-modelo
const inst = b64f(D.inst, Float32Array), NI = D.ninst, W = 11;
const byT = new Map();
for (let i = 0; i < NI; i++){ const t = inst[i*W]; if(!byT.has(t)) byT.set(t,[]); byT.get(t).push(i); }
const aggMat = new THREE.MeshStandardMaterial({roughness:0.92, metalness:0, flatShading:true, clippingPlanes:planes, side:THREE.DoubleSide});
const meshes = [];
const m4 = new THREE.Matrix4(), qq = new THREE.Quaternion(), pp = new THREE.Vector3(), ss = new THREE.Vector3();
byT.forEach((ids, t) => {
  const tp = D.tpl[t];
  const g = new THREE.BufferGeometry();
  g.setAttribute('position', new THREE.BufferAttribute(b64f(tp.v, Float32Array), 3));
  g.setIndex(new THREE.BufferAttribute(new Uint32Array(b64f(tp.f, Uint16Array)), 1));
  g.computeVertexNormals();
  const im = new THREE.InstancedMesh(g, aggMat, ids.length);
  ids.forEach((i, k) => {
    const o = i*W, s = inst[o+1];
    pp.set(inst[o+2], inst[o+3], inst[o+4]); qq.set(inst[o+6], inst[o+7], inst[o+8], inst[o+5]); ss.set(s,s,s);
    im.setMatrixAt(k, m4.compose(pp, qq, ss));
  });
  im.userData.ids = ids; meshes.push(im); model.add(im);
});
const col = new THREE.Color();
function paint(){
  meshes.forEach(im => { im.userData.ids.forEach((i,k)=>{ const o=i*W;
    const c = st.color==='s' ? STONE[inst[o+10]|0] : FRAC[inst[o+9]|0]; col.setRGB(c[0]/255,c[1]/255,c[2]/255).convertSRGBToLinear(); im.setColorAt(k,col); });
    im.instanceColor.needsUpdate = true; });
}

// vazios
const vg = new THREE.BufferGeometry();
vg.setAttribute('position', new THREE.BufferAttribute(b64f(D.void_v, Float32Array), 3));
vg.setIndex(new THREE.BufferAttribute(b64f(D.void_f, Uint32Array), 1));
vg.computeVertexNormals();
const voidMat = new THREE.MeshStandardMaterial({color:new THREE.Color(css('--void')||'#e23d28'), roughness:0.5, clippingPlanes:planes, side:THREE.DoubleSide});
const voidMesh = new THREE.Mesh(vg, voidMat); model.add(voidMesh);

// pele de mastique (cilindro) + tampas texturizadas com a seção real
const shellMat = new THREE.MeshStandardMaterial({color:0x1b1a19, roughness:0.75, clippingPlanes:planes, side:THREE.DoubleSide});
const shell = new THREE.Mesh(new THREE.CylinderGeometry(R, R, H, 160, 1, true), shellMat);
shell.rotation.x = Math.PI/2; shell.position.z = H/2; model.add(shell);
const loader = new THREE.TextureLoader(), texCache = {};
const tex = name => { if(!texCache[name]){ const t = loader.load(window.V5SEC[name], ()=>render()); t.encoding = THREE.sRGBEncoding; t.anisotropy = 4; texCache[name]=t; } return texCache[name]; };
const capMat = () => new THREE.MeshStandardMaterial({transparent:true, alphaTest:0.5, roughness:0.85, side:THREE.DoubleSide});
const topCap = new THREE.Mesh(new THREE.PlaneGeometry(2*R, 2*R), capMat()); topCap.position.z = H; topCap.material.clippingPlanes = planes;
const botCap = new THREE.Mesh(new THREE.PlaneGeometry(2*R, 2*R), capMat()); botCap.position.z = 0; botCap.material.clippingPlanes = planes;
model.add(topCap, botCap);
const cutCapZ = new THREE.Mesh(new THREE.PlaneGeometry(2*R, 2*R), capMat()); model.add(cutCapZ);
const cutCapX = new THREE.Mesh(new THREE.PlaneGeometry(2*R, H), capMat());
const capXHolder = new THREE.Group(); capXHolder.add(cutCapX); model.add(capXHolder);
// o plano vertical vive em (y,z) do modelo: u -> y, v -> z
cutCapX.quaternion.setFromRotationMatrix(new THREE.Matrix4().makeBasis(new THREE.Vector3(0,1,0), new THREE.Vector3(0,0,1), new THREE.Vector3(1,0,0)));

const cutlbl = document.getElementById('cutlbl');
function idx(){ return st.cut==='z' ? Math.round(st.pos/100*(D.zs.length-1)) : Math.round(st.pos/100*(D.xs.length-1)); }
function apply(){
  const real = st.show === 'real';
  shell.visible = real; topCap.visible = real && st.cut !== 'z'; botCap.visible = real;
  topCap.material.map = tex(st.color+'z'+(D.zs.length-1)); topCap.material.needsUpdate = true;
  botCap.material.map = tex(st.color+'z0'); botCap.material.needsUpdate = true;
  voidMesh.visible = st.voids;
  cutCapZ.visible = false; cutCapX.visible = false;
  if (st.cut === 'none'){ clip.normal.set(0,-1,0); clip.constant = 1e4; cutlbl.textContent = '—'; }
  else if (st.cut === 'z'){
    const k = idx(), z = D.zs[k];
    clip.normal.set(0,-1,0); clip.constant = z;            // mantém y_mundo < z
    cutCapZ.position.z = z; cutCapZ.visible = real;
    cutCapZ.material.map = tex(st.color+'z'+k); cutCapZ.material.needsUpdate = true;
    cutlbl.textContent = `z = ${z.toFixed(1).replace('.',',')} mm`;
  } else {
    const k = idx(), x = D.xs[k];
    clip.normal.set(-1,0,0); clip.constant = x;             // mantém x < corte
    capXHolder.position.set(x, 0, H/2); cutCapX.visible = real;
    cutCapX.material.map = tex(st.color+'x'+k); cutCapX.material.needsUpdate = true;
    cutlbl.textContent = `x = ${x.toFixed(0)} mm`;
  }
  document.getElementById('cutpos').disabled = st.cut === 'none';
  paint(); drawLegend(); show2d(); render();
}

// ---------- vista 2D (seções) ----------
const secimg = document.getElementById('secimg');
function show2d(){
  if (st.tab !== '2d') return;
  const c = st.cut === 'none' ? 'x' : st.cut;
  const k = c==='z' ? Math.round(st.pos/100*(D.zs.length-1)) : Math.round(st.pos/100*(D.xs.length-1));
  secimg.src = window.V5SEC[st.color+c+k];
  secimg.alt = c==='z' ? `Seção horizontal a z = ${D.zs[k]} mm` : `Seção vertical a x = ${D.xs[k]} mm`;
}

// ---------- controles ----------
function seg(id, key){ const el = document.getElementById(id);
  el.querySelectorAll('button').forEach(b => b.onclick = () => { st[key] = b.dataset.v;
    el.querySelectorAll('button').forEach(x => x.setAttribute('aria-pressed', x===b)); apply(); }); }
seg('segShow','show'); seg('segColor','color'); seg('segCut','cut');
document.getElementById('cutpos').oninput = e => { st.pos = +e.target.value; apply(); };
document.getElementById('voids').onchange = e => { st.voids = e.target.checked; apply(); };
const t3 = document.getElementById('t3d'), t2 = document.getElementById('t2d');
function tab(t){ st.tab = t; t3.setAttribute('aria-pressed', t==='3d'); t2.setAttribute('aria-pressed', t==='2d');
  host.hidden = t!=='3d'; document.getElementById('view2d').hidden = t!=='2d';
  document.getElementById('segShow').hidden = t!=='3d'; document.getElementById('voids').parentElement.hidden = t!=='3d';
  if (t==='2d' && st.cut==='none'){ st.cut='x'; document.querySelectorAll('#segCut button').forEach(x=>x.setAttribute('aria-pressed', x.dataset.v==='x')); }
  apply(); resize(); }
t3.onclick = () => tab('3d'); t2.onclick = () => tab('2d');

// ---------- render ----------
let pending = false;
function render(){ if (pending) return; pending = true; requestAnimationFrame(()=>{ pending=false; controls.update(); renderer.render(scene, camera); }); }
controls.addEventListener('change', render);
function resize(){ const r = host.getBoundingClientRect(); if(!r.width) return; renderer.setSize(r.width, r.height, false);
  renderer.domElement.style.width='100%'; renderer.domElement.style.height='100%'; camera.aspect = r.width/r.height; camera.updateProjectionMatrix(); render(); }
new ResizeObserver(resize).observe(host);
(function loop(){ if (controls.update()) render(); requestAnimationFrame(loop); })();
document.getElementById('loading').hidden = true;
apply(); resize();

} catch (err) { const l = document.getElementById('loading'); l.hidden = false; l.textContent = 'Erro ao montar o modelo: ' + err.message; console.error(err); }
}
</script>


In [ ]:
%%writefile run.py
"""Uso:  python run.py config_V5.json
Gera a pasta resultado_<nome>/ com o visualizador HTML, os STL e um JSON com os resultados."""
import sys
from md_core import rodar

if __name__ == '__main__':
    rodar(sys.argv[1] if len(sys.argv) > 1 else 'config_V5.json')


## Mistura
Altere os valores abaixo. Para um teste rápido, mude `TESTE_RAPIDO = True` (porção de 250 g, Ø65 × 31,5 mm).

In [ ]:
import json
TESTE_RAPIDO = False

mistura = {
  "nome": "V5",
  "descricao": "SMA 8 · calcário Tunstead · betume 40/60 · Awuah et al. (2024), Tabela 10",
  "granulometria": {
    "peneiras_mm": [
      10,
      8,
      6.3,
      4,
      2.8,
      2,
      1,
      0.5,
      0.25,
      0.125,
      0.063
    ],
    "passante_pct": [
      100.0,
      93.6,
      84.3,
      55.9,
      38.8,
      28.6,
      19.1,
      15.2,
      12.9,
      10.9,
      9.0
    ]
  },
  "teor_betume_pct": 5.0,
  "densidade_agregado": 2.7,
  "densidade_betume": 1.03,
  "vazios_alvo_pct": 4.8,
  "corpo_de_prova": {
    "diametro_mm": 101.6,
    "altura_mm": 63.5
  },
  "forma_agregados": {
    "_comentario": "AR ~ Weibull(escala, forma); h = espessura/Min Feret. Tabela 3 do paper. Cada classe vale até o tamanho indicado (mm de peneira).",
    "classes": [
      {
        "ate_mm": 4,
        "AR_escala": 0.73,
        "AR_forma": 6.7,
        "h": 0.74
      },
      {
        "ate_mm": 8,
        "AR_escala": 0.77,
        "AR_forma": 7.37,
        "h": 0.61
      },
      {
        "ate_mm": 999,
        "AR_escala": 0.75,
        "AR_forma": 6.54,
        "h": 0.61
      }
    ],
    "angulosidade": 0.28
  },
  "calibracao_tamanho": {
    "_comentario": "modo 'NA_250g': ajusta o tamanho das partículas para bater o número de agregados por 250 g (paper V5: 1196). modo 'fator': Min Feret = fator x abertura da peneira (Camsizer da Tabela 3: 1.218).",
    "modo": "NA_250g",
    "NA_250g": 1196,
    "fator": 1.218
  },
  "simulacao": {
    "atrito_despejo": 0.4,
    "atrito_compactacao": 0.12,
    "forca_final_pistao_N": 40,
    "semente": 2024
  },
  "voxel_mm": 0.25,
  "propriedades_medidas": {
    "trabalhabilidade_pct": 62.4,
    "estabilidade_kN": 10.4,
    "fluencia_mm": 3.6,
    "vazios_pct": 4.8
  },
  "propriedades_nomograma": {
    "trabalhabilidade_pct": 66.0,
    "estabilidade_kN": 9.7,
    "fluencia_mm": 3.7,
    "vazios_pct": 4.5
  }
}

if TESTE_RAPIDO:
    mistura['nome'] += '_250g'
    mistura['corpo_de_prova'] = {'diametro_mm': 65.0, 'altura_mm': 31.5}
json.dump(mistura, open('config.json', 'w'), ensure_ascii=False, indent=1)
print('configuração salva:', mistura['nome'])


## Rodar

In [ ]:
import importlib, md_core
importlib.reload(md_core)
pasta = md_core.rodar('config.json')


## Baixar resultados

In [ ]:
import os
from google.colab import files
nome = json.load(open('config.json'))['nome']
files.download(os.path.join(pasta, f'{nome}_visualizador.html'))
files.download(os.path.join(pasta, f'{nome}_STL.zip'))
files.download(os.path.join(pasta, f'{nome}_resultados.json'))
